# 🌲 Step 3 — NFI Integration & Hypsometer Math
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

## What this step does
Turns a raw NFI plot table into a validated, **projected (EPSG:25829)** GeoPackage: applies the
DBH ≥ 10 cm EU filter, computes tree height with the laser-hypsometer equation, and exports both
GeoPackage and Excel. You can upload your own NFI CSV/XLSX in the template schema.

## The Science
Field crews don't climb trees — they sight the prism with a **laser hypsometer** and record three
numbers. Height comes from right-triangle trigonometry:

$$H = D \times \tan(\theta) + h$$

where `D` = slope distance (m), `θ` = sighting angle, `h` = observer eye height. A 20 m sight at
40° with eyes at 1.5 m gives `20·tan(40°) + 1.5 ≈ 18.3 m`. Height × species allometry is what
converts inventory trees into the biomass factors used in Step 5 — so this math is the bridge
between field work and carbon accounting.

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic NFI table (Day-1 ready)
# ============================================================
import numpy as np
import pandas as pd

np.random.seed(42)

def generate_synthetic_nfi(n_plots=60, seed=42):
    """Synthetic NFI plot table - same contract as Step 0's template."""
    rng = np.random.default_rng(seed)
    lon = rng.uniform(-6.40, -6.10, n_plots)
    lat = rng.uniform(52.95, 53.15, n_plots)
    dbh = rng.uniform(4, 45, n_plots)
    species = rng.choice([1, 2, 3], n_plots, p=[0.55, 0.25, 0.20])
    distance = rng.uniform(5, 25, n_plots)
    angle = rng.uniform(15, 55, n_plots)
    obs_height = np.full(n_plots, 1.5)
    return pd.DataFrame({"x": lon.round(6), "y": lat.round(6), "DBH": dbh.round(1),
                         "Species": species, "Distance": distance.round(1),
                         "Angle": angle.round(1), "ObsHeight": obs_height})

nfi_df = generate_synthetic_nfi()
nfi_path = outpath("step3_nfi_raw.csv")
nfi_df.to_csv(nfi_path, index=False)
print(f"Synthetic NFI: {len(nfi_df)} plots -> {nfi_path}")

## Optional: upload your own NFI data
Required columns: `x, y, DBH, Species, Distance, Angle, ObsHeight` (same schema as Step 0).
Upload replaces the synthetic table for the processing below.

In [ ]:
# ============================================================
# Optional: template + upload widget
# ============================================================
REQUIRED_COLUMNS = ["x", "y", "DBH", "Species", "Distance", "Angle", "ObsHeight"]

template_path = outpath("step3_NFI_template.xlsx")
nfi_df.head(3).to_excel(template_path, index=False)
make_download_button(template_path, label="⬇️ Download blank NFI template (.xlsx)",
                     register_as="Step 3 NFI template")

def handle_nfi_upload(fname, content_bytes):
    global nfi_df
    import io
    try:
        candidate = (pd.read_csv(io.BytesIO(content_bytes)) if fname.lower().endswith(".csv")
                     else pd.read_excel(io.BytesIO(content_bytes)))
    except Exception as e:
        print(f"❌ Could not parse '{fname}': {e}")
        return
    missing = [c for c in REQUIRED_COLUMNS if c not in candidate.columns]
    if missing:
        print(f"❌ Upload rejected - missing required column(s): {missing}. Synthetic data stays active.")
        return
    nfi_df = candidate.copy()
    print(f"✅ SUCCESS: loaded {len(nfi_df)} plots from '{fname}'.")

make_upload_widget(handle_nfi_upload, accept=".csv,.xlsx", label="Upload NFI .csv/.xlsx")
print("Skip to continue with the synthetic table.")

In [ ]:
# ============================================================
# Process: DBH filter + hypsometer height + reproject to EPSG:25829
# ============================================================
import geopandas as gpd
import matplotlib.pyplot as plt

def calculate_tree_height(distance_m, angle_deg, observer_height_m):
    """Laser hypsometer equation: H = D*tan(theta) + h_eye."""
    return distance_m * np.tan(np.radians(angle_deg)) + observer_height_m

def process_nfi(nfi_df):
    valid = nfi_df[nfi_df["DBH"] >= 10.0].copy()
    valid["Height"] = calculate_tree_height(valid["Distance"], valid["Angle"], valid["ObsHeight"])
    gdf = gpd.GeoDataFrame(valid, geometry=gpd.points_from_xy(valid["x"], valid["y"]),
                           crs="EPSG:4326")
    return gdf.to_crs("EPSG:25829"), len(nfi_df) - len(valid)

gdf_utm, n_rejected = process_nfi(nfi_df)
print(f"DBH filter: kept {len(gdf_utm)} / {len(nfi_df)} plots ({n_rejected} rejected)")
print(f"Height range: {gdf_utm['Height'].min():.1f} - {gdf_utm['Height'].max():.1f} m")
gdf_utm[["DBH", "Species", "Distance", "Angle", "Height"]].head()

In [ ]:
# ============================================================
# Visualise: projected plot map + height vs DBH
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].scatter(gdf_utm.geometry.x, gdf_utm.geometry.y, c=gdf_utm["Species"],
                cmap="viridis", s=30, edgecolor="k", linewidth=0.3)
axes[0].set_title("Validated NFI plots in EPSG:25829 (metres)")
axes[0].set_xlabel("Easting (m)"); axes[0].set_ylabel("Northing (m)")
axes[1].scatter(gdf_utm["DBH"], gdf_utm["Height"], c=gdf_utm["Species"], cmap="viridis", s=30)
axes[1].set_title("Hypsometer height vs DBH (colored by species)")
axes[1].set_xlabel("DBH (cm)"); axes[1].set_ylabel("Height (m)")
plt.tight_layout()

nfi_png = outpath("step3_nfi_processing.png")
fig.savefig(nfi_png, dpi=120, bbox_inches="tight")
plt.show()

make_download_button(nfi_png, label="⬇️ Download processing chart (.png)", register_as="Step 3 chart")

# Export GeoPackage (fallback to GeoJSON if the GPKG driver is missing)
gpkg_path = outpath("step3_nfi_validated.gpkg")
try:
    gdf_utm.to_file(gpkg_path, driver="GPKG")
    make_download_button(gpkg_path, label="⬇️ Download validated plots (.gpkg)",
                         register_as="Step 3 validated NFI (GPKG)")
except Exception as e:
    geojson_path = outpath("step3_nfi_validated.geojson")
    gdf_utm.to_file(geojson_path, driver="GeoJSON")
    print(f"GPKG driver unavailable ({e}) - exported GeoJSON instead.")
    make_download_button(geojson_path, label="⬇️ Download validated plots (.geojson)",
                         register_as="Step 3 validated NFI (GeoJSON)")

xlsx_path = outpath("step3_nfi_heights.xlsx")
gdf_utm.drop(columns="geometry").to_excel(xlsx_path, index=False)
make_download_button(xlsx_path, label="⬇️ Download heights table (.xlsx)",
                     register_as="Step 3 heights table")

**Interpretation:** the left map shows the plots after reprojection — the axes are now **metres
eastings/northings**, so distance and area calculations against satellite pixels are valid. The
right panel is the hypsometer equation at work: height climbs with DBH, and the spread between
species (colors) is exactly the species-specific allometry that Step 5's biomass matrix encodes.
The rejected sub-10 cm plots never made it here — the fail-loud filter removed them upstream.

In [ ]:
# ============================================================
# Regenerate control
# ============================================================
def rerun_step3(seed=None):
    """Fresh synthetic NFI + full processing chain (filter, heights, reprojection)."""
    global nfi_df, gdf_utm, n_rejected
    seed = seed or np.random.randint(0, 10_000)
    nfi_df = generate_synthetic_nfi(seed=seed)
    gdf_utm, n_rejected = process_nfi(nfi_df)
    print(f"[seed={seed}] {len(nfi_df)} plots -> {len(gdf_utm)} validated")

make_regenerate_button(rerun_step3, label="🔄 Regenerate NFI & re-run processing")
make_download_all_button()